# SEC Filings Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kineviz/fortune500/blob/main/pipeline.ipynb)

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud Authentication** – Authenticate and set active GCP project
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for inspection)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7)
5. **Upload JSONL to GCS** – Sync extracted sections to Cloud Storage
6. **Initialize BigQuery + Load Sections** – Create core tables and load sections JSONL
7. **Run BigQuery Extraction** – Generate `insights` via `AI.GENERATE_TEXT`
8. **Export Insights + Python Normalization** – Transform to parquet and run entity normalization
9. **Upload Parquet to GCS** – Sync generated node/edge parquet files
10. **Load Parquet Tables to BigQuery** – Load graph node/edge tables (skip missing optional outputs)
11. **Create Property Graph DDL** – Build `SecGraph` in BigQuery


### 0.0 GCP Environment Setup Checklist
Before running this notebook, ensure your Google Cloud environment is fully configured:

1.  **API Enablement**: Enable [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com).
2.  **Billing**: Ensure your project is linked to an active **[Billing Account](https://console.cloud.google.com/billing/enable)**.
3.  **BigQuery AI Connection**:
    - Create a Cloud Resource Connection named **`vertex_ai_connection`** in the **US** location.
    - Grant the connection's Service Account the **`roles/aiplatform.user`** role.
    - **Run the cell below** to automate this setup.

## Configuration

Set the environment constants below. These variables control GCP authentication, local and cloud storage, and pipeline depth.

### GCP Settings
- **`GCP_PROJECT`**: Your Google Cloud Project ID. Find this in the [GCP Console Project Dashboard](https://console.cloud.google.com/home/dashboard). Ensure you have the [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com) enabled.
- **`GCS_BUCKET`**: The Google Cloud Storage bucket used for staging data for BigQuery load. Format: `gs://your-bucket-name`.
- **`BQ_LOCATION`**: The regional location for your BigQuery datasets (e.g., `US`, `EU`).
- **`BQ_DATASET`**: The name of the BigQuery Dataset (default: `sec_filings`). Allows isolating runs.
- **`GEMINI_MODEL`**: The Gemini model version used for text extraction (e.g., `gemini-3.5-pro`).
- **`BQ_MODEL`**: The database identifier name assigned to the LLM model connection (default: `gemini_pro_latest`).
- **Billing**: BigQuery AI functions require an active billing account. Ensure it is enabled [here](https://console.cloud.google.com/billing/enable).

### Local Directories
These variables define where data is stored on your local machine or external drive.
- **`SGML_DIR`**: Root directory for raw `.sgml` files from SEC.gov.
- **`MARKDOWN_DIR`**: Directory for converted `.md` files.
- **`JSON_DIR`**: Directory for sectioned JSONL files (Item 1, 1A, 7).

### Scraper Settings
- **`SCRAPER_LIMIT`**: The number of companies to pull from the companies list (e.g., 5, 20, 500). Use a small number to test.
- **`SCRAPER_LAST_N_YEARS`**: The number of historical years to scrape for each company.

In [ ]:
# Assert configuration is updated
from IPython import get_ipython

# Default to invalid until checks pass
_CONFIG_VALIDATED = False

def _block_until_config_validated(_info):
    if not globals().get("_CONFIG_VALIDATED", False):
        raise RuntimeError(
            "Configuration is invalid. Re-run the Configuration cell and fix all errors before running other cells."
        )

ip = get_ipython()
if ip is not None:
    # Register once per kernel session
    if not globals().get("_CONFIG_GUARD_REGISTERED", False):
        ip.events.register("pre_run_cell", _block_until_config_validated)
        _CONFIG_GUARD_REGISTERED = True

errors = []
if GCP_PROJECT == "YOUR_PROJECT_ID":
    errors.append("Update GCP_PROJECT at the top of the notebook.")
if GCS_BUCKET == "gs://YOUR_BUCKET":
    errors.append("Update GCS_BUCKET at the top of the notebook.")
if "-" in BQ_DATASET:
    errors.append("BQ_DATASET cannot contain '-' (use underscores instead).")

if errors:
    raise RuntimeError("Configuration validation failed:\n- " + "\n- ".join(errors))

_CONFIG_VALIDATED = True
print("✓ Configuration validated")


## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
from IPython import get_ipython
ipy = get_ipython()

if "google.colab" in sys.modules:
    def has_repo(path):
        return os.path.exists(os.path.join(path, "01_scraper.py"))

    cwd = os.getcwd()
    candidates = [
        cwd,
        os.path.join(cwd, "fortune500"),
        "/content/fortune500",
        "/content/fortune500/fortune500",
    ]
    repo_root = next((p for p in candidates if has_repo(p)), None)

    if repo_root is None:
        if not os.path.exists("/content/fortune500"):
            ipy.system("git clone https://github.com/Kineviz/fortune500.git /content/fortune500")
        repo_root = "/content/fortune500"

    os.chdir(repo_root)
    print(f"Colab repo root: {os.getcwd()}")

ipy.system("pip install -r requirements.txt")


## 1. Setup & Google Cloud Authentication

In [ ]:
# Using GCP_PROJECT from configuration above

if GCP_PROJECT != "YOUR_PROJECT_ID":
    from IPython import get_ipython
    import subprocess
    import google.auth
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")

    # In Colab, this sets up Application Default Credentials for Python clients
    if 'google.colab' in sys.modules:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
    else:
        ipy.system("gcloud auth login")
        ipy.system("gcloud auth application-default login")

    result = subprocess.run(["gcloud", "config", "get-value", "project"], capture_output=True, text=True)
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")

    # Verify ADC is available to BigQuery Python client
    creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    print(f"✓ ADC ready (quota project: {getattr(creds, 'quota_project_id', None) or adc_project or GCP_PROJECT})")
else:
    raise ValueError("Set GCP_PROJECT in the Configuration cell at the beginning of the notebook.")

In [ ]:
import subprocess
import os
import sys
import json

def check_billing(project_id):
    print(f"Checking billing status for {project_id}...")
    try:
        # Using gcloud to verify if billing is enabled on the given project
        result = subprocess.run(
            ["gcloud", "billing", "projects", "describe", project_id, "--format=json"],
            capture_output=True, text=True, check=True
        )
        billing_info = json.loads(result.stdout)
        if billing_info.get("billingEnabled"):
            print("\u2705 Billing is enabled! You are ready to use BigQuery AI.")
        else:
            print("\u274c ERROR: Billing is NOT enabled for this project.")
            print("BigQuery AI will not function until billing is enabled in the Google Cloud Console.")
            print(f"Visit: https://console.cloud.google.com/billing/enable?project={project_id}")
            raise Exception("Billing not enabled")
    except subprocess.CalledProcessError as e:
        print("\u26a0 Could not verify billing status automatically. Output:")
        print(e.stderr)
        print("\nEnsure you are authenticated and have permissions (roles/billing.projectManager).")

if GCP_PROJECT and GCP_PROJECT != "YOUR_PROJECT_ID":
    check_billing(GCP_PROJECT)


### Ensure we're in the project root (where 01_scraper.py, SQL files, list.csv live)


In [ ]:
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "01_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

# Create local data directories if they don't exist
for d in [SGML_DIR, MARKDOWN_DIR, JSON_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Verified directories: {SGML_DIR}, {MARKDOWN_DIR}, {JSON_DIR}")

### Verify GCS bucket exists and is accessible


In [ ]:
import subprocess
import os
import sys

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "01_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

## 2. Run Scraper (01_scraper.py)

In [ ]:
# Run scraper for explicit tickers (if provided), otherwise use list.csv + SCRAPER_LIMIT
tickers = [t.strip().upper() for t in (TICKERS or []) if str(t).strip()]

if tickers:
    print(f"Scraping explicit tickers: {tickers} ({SCRAPER_LAST_N_YEARS} years each)...")
    for t in tickers:
        scraper = scraper_mod.SECScraper(
            ticker=t,
            last_n_years=SCRAPER_LAST_N_YEARS,
            workers=5,
            output_dir=SCRAPER_OUTPUT,
            dry_run=False,
        )
        await scraper.run()  # Jupyter has its own event loop
else:
    scraper = scraper_mod.SECScraper(
        limit=SCRAPER_LIMIT,
        last_n_years=SCRAPER_LAST_N_YEARS,
        workers=5,
        output_dir=SCRAPER_OUTPUT,
        dry_run=False,
    )

    # Scraper has internal tqdm; get expected count for context
    import pandas as pd
    try:
        n_companies = min(SCRAPER_LIMIT, len(pd.read_csv("list.csv")))
        print(f"Scraping up to {n_companies} companies ({SCRAPER_LAST_N_YEARS} years each)...")
    except FileNotFoundError:
        n_companies = SCRAPER_LIMIT
    await scraper.run()  # Jupyter has its own event loop


In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

## 3. Run Parser (02_parser.py) (Optional)

In [ ]:
# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Parsing SGML to Markdown", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "02_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 4. Extract Sections (03_extract_sections.py)

In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Extracting sections to JSONL", unit="section") as pbar:
    subprocess.run(
        [sys.executable, "03_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 5. Upload JSONL to GCS

In [ ]:
import os
import glob
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
json_files = glob.glob(os.path.join(json_src, "*", "*", "sections.jsonl"))
n_json = len(json_files)
if n_json == 0:
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    with tqdm(total=n_json, desc="Uploading JSONL to GCS", unit="file") as pbar:
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
        pbar.update(n_json)
    print(f"✓ Synced {n_json} section files to {GCS_BUCKET}/json")

## 6. BigQuery Extraction & Python Normalization Pipeline

This hybrid pipeline uses BigQuery's native `AI.GENERATE_TEXT` to extract insights, then exports them to run the Python entity normalization locally.

### 6.1 Initialize Tables and Load Sections

Initialize the `sections` and `insights` tables in BigQuery, and load the raw sections from GCS.

In [ ]:
import subprocess
import google.auth
from google.cloud import bigquery

if 'google.colab' in __import__('sys').modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=GCP_PROJECT, credentials=creds)
print(f"✓ BigQuery client ready (ADC project: {adc_project or GCP_PROJECT})")

def run_bq_query(filename):
    with open(filename, 'r') as f:
        sql = f.read()
    sql = sql.replace('sec_filings.', f"{BQ_DATASET}.")
    sql = sql.replace('sec_filings;', f"{BQ_DATASET};")
    print(f"Executing {filename}...")
    job = client.query(sql)
    job.result()
    print(f"✓ Executed {filename}")

run_bq_query("04_init_tables.sql")

print("Loading sections into BigQuery...")
sections_schema = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"
sections_glob = f"{GCS_BUCKET}/json/*/*/sections.jsonl"

# Resolve explicit section file URIs for BigQuery load
ls_cmd = ["gsutil", "ls", sections_glob]
ls_proc = subprocess.run(ls_cmd, capture_output=True, text=True)
if ls_proc.returncode != 0:
    print("No sections found in GCS at:", sections_glob)
    local_json_dir = globals().get('JSON_DIR', './data/json')
    print(f"Attempting upload from local {local_json_dir} -> {GCS_BUCKET}/json ...")
    subprocess.run(["gsutil", "-m", "rsync", "-r", local_json_dir, f"{GCS_BUCKET}/json"], check=True)
    ls_proc = subprocess.run(ls_cmd, capture_output=True, text=True)
    if ls_proc.returncode != 0:
        raise FileNotFoundError(
            f"No sections.jsonl found at {sections_glob}. Run extraction (Section 4) and upload (Section 5) first."
        )

section_uris = [u.strip() for u in ls_proc.stdout.splitlines() if u.strip().endswith("sections.jsonl")]
if not section_uris:
    raise FileNotFoundError(f"No concrete sections.jsonl files found for pattern: {sections_glob}")
print(f"✓ Found {len(section_uris)} section files in GCS")

# Scope load strictly to the current local extraction run
from pathlib import Path
local_json_dir = Path(globals().get('JSON_DIR', './data/json'))
local_sections = sorted(local_json_dir.glob('*/*/sections.jsonl'))
if local_sections:
    expected_uris = {
        f"{GCS_BUCKET}/json/{p.parent.parent.name}/{p.parent.name}/sections.jsonl"
        for p in local_sections
    }
    scoped_uris = [u for u in section_uris if u in expected_uris]
    if not scoped_uris:
        raise FileNotFoundError(
            f"No matching GCS section files for current local run under {local_json_dir}. "
            "Run extraction/upload again before BigQuery load."
        )
    section_uris = scoped_uris
    print(f"✓ Scoped load to {len(section_uris)} files from current local run")
else:
    print(f"⚠ No local sections found under {local_json_dir}; using all matched GCS files")

type_map = {"STRING": "STRING", "INTEGER": "INT64"}
schema = []
for col in sections_schema.split(','):
    name, typ = col.split(':')
    schema.append(bigquery.SchemaField(name, type_map.get(typ, typ)))

table_id = f"{GCP_PROJECT}.{BQ_DATASET}.sections"
load_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(section_uris, table_id, job_config=load_config)
load_job.result()
print("✓ Sections loaded")


### 6.2 Run AI Extraction in BigQuery

Extract insights using `AI.GENERATE_TEXT`.

In [ ]:
run_bq_query("05_extraction.sql")

### 6.3 Export Insights and Run Python Normalization

Export the `insights` table to GCS, download it locally, and run the Python transformation and normalization scripts.

In [ ]:
import os
import subprocess
import pandas as pd
from pathlib import Path

# Keep BigQuery export/import paths and python_pipeline output in sync
PIPELINE_MODEL_NAME = GEMINI_MODEL

print("Exporting insights to GCS...")
subprocess.run([
    "bq", "extract", "--destination_format=NEWLINE_DELIMITED_JSON",
    f"{GCP_PROJECT}:{BQ_DATASET}.insights",
    f"{GCS_BUCKET}/parquets/{PIPELINE_MODEL_NAME}/insights.jsonl"
], check=True)

print("Downloading insights locally...")
local_insights_dir = f"python_pipeline/output/{PIPELINE_MODEL_NAME}/extractions"
os.makedirs(local_insights_dir, exist_ok=True)
subprocess.run([
    "gsutil", "cp",
    f"{GCS_BUCKET}/parquets/{PIPELINE_MODEL_NAME}/insights.jsonl",
    f"{local_insights_dir}/insights.jsonl"
], check=True)

print("Preparing python_pipeline environment...")
subprocess.run(["uv", "sync", "--extra", "vertex"], cwd="python_pipeline", check=True)

py_env = os.environ.copy()
py_env["LLM_PROVIDER"] = "vertex"
py_env["MODEL_NAME"] = PIPELINE_MODEL_NAME
py_env["VERTEX_PROJECT"] = GCP_PROJECT
py_env["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT
py_env["GCS_BUCKET"] = GCS_BUCKET
py_env["DATA_DIR"] = JSON_DIR
py_env["OUTPUT_DIR"] = "output"


import base64

def ensure_entity_norm_taxonomy_files():
    data_dir = Path("python_pipeline/entity_normalization/data")
    data_dir.mkdir(parents=True, exist_ok=True)
    embedded = {
        "competitor_sectors.csv": "c2VjdG9yLGRlc2NyaXB0aW9uLGtleXdvcmRzLGV4YW1wbGVfY29tcGFuaWVzLGV4YW1wbGVfY2F0ZWdvcmllcwpUZWNobm9sb2d5LCJTb2Z0d2FyZSwgaGFyZHdhcmUsIHNlbWljb25kdWN0b3JzLCBjbG91ZCBjb21wdXRpbmcsIEFJL01MLCBpbnRlcm5ldCBzZXJ2aWNlcywgY3liZXJzZWN1cml0eSwgZGF0YSBjZW50ZXJzLCBJVCBzZXJ2aWNlcyIsInRlY2hub2xvZ3ksIHRlY2gsIHNvZnR3YXJlLCBTYWFTLCBjbG91ZCwgQUksIHNlbWljb25kdWN0b3IsIGNoaXAsIGNvbXB1dGluZywgaW50ZXJuZXQsIGRpZ2l0YWwsIGN5YmVyLCBkYXRhIGNlbnRlciwgSVQgc2VydmljZXMsIHBsYXRmb3JtLCBhcHAsIHByb2Nlc3NvciwgR1BVLCBzZXJ2ZXIsIGRhdGFiYXNlIiwiQXBwbGUsIE1pY3Jvc29mdCwgTlZJRElBLCBHb29nbGUsIEludGVsLCBDaXNjbywgU2FsZXNmb3JjZSwgT3JhY2xlLCBBTUQsIFF1YWxjb21tIiwiQ2xvdWQgU2VydmljZSBQcm92aWRlcnMsIFNvZnR3YXJlIERldmVsb3BtZW50IEZpcm1zLCBTZW1pY29uZHVjdG9yIEVxdWlwbWVudCBNYW51ZmFjdHVyZXJzLCBTYWFTIFBsYXRmb3JtcyIKRmluYW5jaWFsIFNlcnZpY2VzLCJCYW5raW5nLCBpbnN1cmFuY2UsIGFzc2V0IG1hbmFnZW1lbnQsIGZpbnRlY2gsIHBheW1lbnRzLCBsZW5kaW5nLCBicm9rZXJhZ2UsIHByaXZhdGUgZXF1aXR5LCByZWluc3VyYW5jZSIsImZpbmFuY2lhbCwgYmFua2luZywgYmFuaywgaW5zdXJhbmNlLCBhc3NldCBtYW5hZ2UsIGludmVzdG1lbnQsIGZpbnRlY2gsIGNyZWRpdCwgbGVuZGluZywgbW9ydGdhZ2UsIGJyb2tlcmFnZSwgd2VhbHRoLCBoZWRnZSBmdW5kLCBwcml2YXRlIGVxdWl0eSwgcmVpbnN1cmFuY2UsIHBheW1lbnRzLCB1bmRlcndyaXRpbmcsIGN1c3RvZHkiLCJKUE1vcmdhbiBDaGFzZSwgVmlzYSwgTWFzdGVyY2FyZCwgQmVya3NoaXJlIEhhdGhhd2F5LCBHb2xkbWFuIFNhY2hzLCBCbGFja1JvY2ssIEFJRywgUHJ1ZGVudGlhbCIsIkZpbnRlY2ggQ29tcGFuaWVzLCBQcmltZSBCcm9rZXJzLCBJbnN1cmFuY2UgQ2FycmllcnMsIFJlZ2lvbmFsIEJhbmtzLCBBc3NldCBNYW5hZ2VycyIKSGVhbHRoY2FyZSAmIFBoYXJtYSwiUGhhcm1hY2V1dGljYWxzLCBiaW90ZWNobm9sb2d5LCBtZWRpY2FsIGRldmljZXMsIGhvc3BpdGFscywgaGVhbHRoIHNlcnZpY2VzLCBkaWFnbm9zdGljcywgbGlmZSBzY2llbmNlcyIsImhlYWx0aCwgcGhhcm1hLCBkcnVnLCBiaW90ZWNoLCBtZWRpY2FsLCBob3NwaXRhbCwgY2xpbmljYWwsIHRoZXJhcGV1dGljLCBiaW9zaW1pbGFyLCBnZW5lcmljLCBGREEsIHBhdGllbnQsIGRlbnRhbCwgZGlhZ25vc3RpYywgbGlmZSBzY2llbmNlLCBzdXJnaWNhbCwgb25jb2xvZ3kiLCJQZml6ZXIsIEpvaG5zb24gJiBKb2huc29uLCBVbml0ZWRIZWFsdGgsIE1lZHRyb25pYywgQXN0cmFaZW5lY2EsIEFiYm90dCwgTWVyY2ssIEFtZ2VuIiwiR2VuZXJpYyBNYW51ZmFjdHVyZXJzLCBNZWRpY2FsIERldmljZSBDb21wYW5pZXMsIFBoYXJtYWN5IEJlbmVmaXQgTWFuYWdlcnMsIEhlYWx0aGNhcmUgUHJvdmlkZXJzIgpFbmVyZ3ksIk9pbCAmIGdhcyAodXBzdHJlYW0vbWlkc3RyZWFtL2Rvd25zdHJlYW0pLCByZW5ld2FibGUgZW5lcmd5LCBwb3dlciBnZW5lcmF0aW9uLCBlbmVyZ3kgdHJhZGluZywgdXRpbGl0aWVzIG92ZXJsYXAgZm9yIGdlbmVyYXRpb24iLCJlbmVyZ3ksIG9pbCwgZ2FzLCBwZXRyb2xldW0sIHJlZmluLCBwaXBlbGluZSwgTE5HLCBPUEVDLCBzb2xhciwgd2luZCwgbnVjbGVhciwgZnVlbCwgcG93ZXIgZ2VuZXJhdCwgdXBzdHJlYW0sIG1pZHN0cmVhbSwgZG93bnN0cmVhbSwgZHJpbGxpbmcsIGV4cGxvcmF0aW9uLCBiaW9mdWVsLCBoeWRyb2dlbiIsIkV4eG9uTW9iaWwsIENoZXZyb24sIFNoZWxsLCBCUCwgQ29ub2NvUGhpbGxpcHMsIE5leHRFcmEgRW5lcmd5LCBFbmJyaWRnZSwgT1BFQysiLCJNYWpvciBJbnRlZ3JhdGVkIE9pbCBDb21wYW5pZXMsIEluZGVwZW5kZW50IFBvd2VyIFByb2R1Y2VycywgUmVuZXdhYmxlIEVuZXJneSBEZXZlbG9wZXJzLCBMTkcgU3VwcGxpZXJzIgpSZXRhaWwgJiBFLWNvbW1lcmNlLCJQaHlzaWNhbCBhbmQgb25saW5lIHJldGFpbCwgZ3JvY2VyeSwgd2hvbGVzYWxlLCBkZXBhcnRtZW50IHN0b3Jlcywgc3BlY2lhbHR5IHJldGFpbCwgZS1jb21tZXJjZSBwbGF0Zm9ybXMiLCJyZXRhaWwsIGUtY29tbWVyY2UsIGVjb21tZXJjZSwgc3RvcmUsIHNob3AsIG1hbGwsIGdyb2NlcnksIHN1cGVybWFya2V0LCB3aG9sZXNhbGUsIG9ubGluZSByZXRhaWwsIG9tbmljaGFubmVsLCBkZXBhcnRtZW50IHN0b3JlLCBkaXNjb3VudCwgb2ZmLXByaWNlLCBjb252ZW5pZW5jZSwgd2FyZWhvdXNlIGNsdWIiLCJXYWxtYXJ0LCBBbWF6b24sIFRhcmdldCwgQ29zdGNvLCBIb21lIERlcG90LCBLcm9nZXIsIDctRWxldmVuLCBEb2xsYXIgR2VuZXJhbCIsIk9ubGluZSBSZXRhaWxlcnMsIEJpZy1Cb3ggUmV0YWlsZXJzLCBDb252ZW5pZW5jZSBTdG9yZXMsIFdhcmVob3VzZSBDbHVicywgRGlzY291bnQgU3RvcmVzIgpJbmR1c3RyaWFsICYgTWFudWZhY3R1cmluZywiSGVhdnkgaW5kdXN0cnksIG1hY2hpbmVyeSwgZXF1aXBtZW50LCBjb25zdHJ1Y3Rpb24sIG1ldGFscywgbWluaW5nLCBidWlsZGluZyBtYXRlcmlhbHMsIHRvb2xzLCBwYWNrYWdpbmciLCJpbmR1c3RyaWFsLCBtYW51ZmFjdHVyLCBzdGVlbCwgZXF1aXBtZW50LCBtYWNoaW5lcnksIGNvbnN0cnVjdGlvbiwgbWluaW5nLCBjZW1lbnQsIGJ1aWxkaW5nIG1hdGVyaWFsLCB0b29saW5nLCBwYWNrYWdpbmcsIHZhbHZlLCBwdW1wLCBjb21wcmVzc29yLCB3ZWxkaW5nLCBmb3JnaW5nIiwiQ2F0ZXJwaWxsYXIsIERlZXJlICYgQ29tcGFueSwgM00sIEhvbmV5d2VsbCwgUGFya2VyIEhhbm5pZmluLCBJbGxpbm9pcyBUb29sIFdvcmtzLCBOdWNvciIsIk9yaWdpbmFsIEVxdWlwbWVudCBNYW51ZmFjdHVyZXJzIChPRU1zKSwgU3RlZWwgUHJvZHVjZXJzLCBTY3JhcCBQcm9jZXNzb3JzLCBCdWlsZGluZyBNYXRlcmlhbHMgU3VwcGxpZXJzIgpDb25zdW1lciBHb29kcywiRm9vZCAmIGJldmVyYWdlLCBob3VzZWhvbGQgcHJvZHVjdHMsIHBlcnNvbmFsIGNhcmUsIGFwcGFyZWwsIHRvYmFjY28sIHBldCBwcm9kdWN0cywgbHV4dXJ5IGdvb2RzIiwiY29uc3VtZXIsIGZvb2QsIGJldmVyYWdlLCBwZXJzb25hbCBjYXJlLCBob3VzZWhvbGQsIHNuYWNrLCBjZXJlYWwsIGNvc21ldGljLCBhcHBhcmVsLCBmYXNoaW9uLCBsdXh1cnksIHBldCwgdG9iYWNjbywgYWxjb2hvbCwgYmVlciwgd2luZSwgc3Bpcml0cywgZGFpcnksIG1lYXQsIGNvbmZlY3Rpb24iLCJOZXN0bGUsIFByb2N0ZXIgJiBHYW1ibGUsIENvY2EtQ29sYSwgUGVwc2lDbywgTmlrZSwgUGhpbGlwIE1vcnJpcywgQ29sZ2F0ZS1QYWxtb2xpdmUsIEtyYWZ0IEhlaW56IiwiUHJpdmF0ZSBMYWJlbCBCcmFuZHMsIEZhc3QgQ2FzdWFsIFJlc3RhdXJhbnRzLCBRdWljay1TZXJ2aWNlIFJlc3RhdXJhbnRzLCBTcGVjaWFsdHkgQXBwYXJlbCBSZXRhaWxlcnMiCkF1dG9tb3RpdmUgJiBUcmFuc3BvcnRhdGlvbiwiVmVoaWNsZXMsIGF1dG8gcGFydHMsIHRydWNraW5nLCBsb2dpc3RpY3MsIHNoaXBwaW5nLCBhaXJsaW5lcywgcmFpbCwgcmlkZS1oYWlsaW5nLCBlbGVjdHJpYyB2ZWhpY2xlcyIsImF1dG8sIHZlaGljbGUsIGNhciwgdHJ1Y2ssIHRyYW5zcG9ydCwgbG9naXN0aWNzLCBzaGlwcGluZywgZnJlaWdodCwgYWlybGluZSwgYXZpYXRpb24sIHJhaWwsIGZsZWV0LCByaWRlLWhhaWwsIGVsZWN0cmljIHZlaGljbGUsIEVWLCBPRU0sIGRlYWxlcnNoaXAsIHRpcmUsIGVuZ2luZSwgY2hhc3NpcyIsIlRveW90YSwgRm9yZCwgVGVzbGEsIEdNLCBCb2VpbmcsIFVQUywgRmVkRXgsIERlbHRhIEFpciBMaW5lcywgVWJlciwgU3RlbGxhbnRpcyIsIkxvdy1Db3N0IENhcnJpZXJzLCBNb3RvciBDYXJyaWVycywgRnJhbmNoaXNlZCBEZWFsZXJzaGlwcywgTFRMIENhcnJpZXJzLCBBdXRvIFBhcnRzIENoYWlucyIKQWVyb3NwYWNlICYgRGVmZW5zZSwiTWlsaXRhcnkgZXF1aXBtZW50LCBkZWZlbnNlIGNvbnRyYWN0aW5nLCBzcGFjZSBzeXN0ZW1zLCBzYXRlbGxpdGVzLCBpbnRlbGxpZ2VuY2Ugc2VydmljZXMiLCJhZXJvc3BhY2UsIGRlZmVuc2UsIGRlZmVuY2UsIG1pbGl0YXJ5LCBzcGFjZSwgbWlzc2lsZSwgc2F0ZWxsaXRlLCB3ZWFwb25zLCBpbnRlbGxpZ2VuY2UsIGZpZ2h0ZXIsIG5hdmFsLCB1bm1hbm5lZCwgcmFkYXIiLCJMb2NraGVlZCBNYXJ0aW4sIE5vcnRocm9wIEdydW1tYW4sIEdlbmVyYWwgRHluYW1pY3MsIFJheXRoZW9uLCBCQUUgU3lzdGVtcywgTDNIYXJyaXMsIEFpcmJ1cyIsIlByaW1lIENvbnRyYWN0b3JzLCBEZWZlbnNlIFN1YmNvbnRyYWN0b3JzLCBHb3Zlcm5tZW50IENvbnRyYWN0b3JzLCBVbm1hbm5lZCBTeXN0ZW1zIFByb3ZpZGVycyIKVGVsZWNvbW11bmljYXRpb25zICYgTWVkaWEsIldpcmVsZXNzL3dpcmVsaW5lIHRlbGVjb21zLCBjYWJsZSwgYnJvYWRiYW5kLCBtZWRpYSBwcm9kdWN0aW9uLCBzdHJlYW1pbmcsIGdhbWluZywgYnJvYWRjYXN0aW5nIiwidGVsZWNvbSwgd2lyZWxlc3MsIGJyb2FkYmFuZCwgY2FibGUsIG1lZGlhLCBicm9hZGNhc3QsIHN0cmVhbWluZywgY29udGVudCwgZW50ZXJ0YWlubWVudCwgZ2FtaW5nLCBzdHVkaW8sIG11c2ljLCB2aWRlbywgVFYsIHRlbGV2aXNpb24sIDVHLCBzcGVjdHJ1bSwgbW9iaWxlLCBmaWJlciIsIkFUJlQsIFZlcml6b24sIFQtTW9iaWxlLCBDb21jYXN0LCBEaXNuZXksIE5ldGZsaXgsIFdhcm5lciBCcm9zLiBEaXNjb3ZlcnksIEFjdGl2aXNpb24gQmxpenphcmQiLCJXaXJlbGVzcyBCcm9hZGJhbmQgUHJvdmlkZXJzLCBTdHJlYW1pbmcgU2VydmljZXMsIENhYmxlIE9wZXJhdG9ycywgTVZQRHMsIEdhbWluZyBPcGVyYXRvcnMiClV0aWxpdGllcywiUmVndWxhdGVkIGVsZWN0cmljL2dhcy93YXRlciBkaXN0cmlidXRpb24sIHRyYW5zbWlzc2lvbiwgZ3JpZCBvcGVyYXRpb25zIChkaXN0aW5jdCBmcm9tIGVuZXJneSBnZW5lcmF0aW9uKSIsInV0aWxpdHksIGVsZWN0cmljIGRpc3RyaWJ1dGlvbiwgd2F0ZXIsIGdhcyBkaXN0cmlidXRpb24sIHRyYW5zbWlzc2lvbiwgcmVndWxhdGVkLCByYXRlcGF5ZXIsIGdyaWQsIElTTywgUlRPLCBtdW5pY2lwYWwgdXRpbGl0eSwgY29vcGVyYXRpdmUsIG1ldGVyaW5nIiwiRHVrZSBFbmVyZ3ksIFNvdXRoZXJuIENvbXBhbnksIEV4ZWxvbiwgRG9taW5pb24gRW5lcmd5LCBQRyZFLCBBRVMiLCJBbHRlcm5hdGl2ZSBFbGVjdHJpYyBTdXBwbGllcnMsIERpc3RyaWJ1dGVkIEdlbmVyYXRpb24gUHJvdmlkZXJzLCBNdW5pY2lwYWwgVXRpbGl0aWVzLCBSZXRhaWwgRWxlY3RyaWMgUHJvdmlkZXJzIgpSZWFsIEVzdGF0ZSwiUkVJVHMsIHByb3BlcnR5IGRldmVsb3BtZW50LCBob3VzaW5nLCBvZmZpY2UvcmV0YWlsL2luZHVzdHJpYWwgc3BhY2UsIGhvc3BpdGFsaXR5IHByb3BlcnRpZXMiLCJyZWFsIGVzdGF0ZSwgUkVJVCwgcHJvcGVydHksIGhvdXNpbmcsIGhvbWVidWlsZGVyLCByZW50YWwsIG9mZmljZSBzcGFjZSwgY29tbWVyY2lhbCByZWFsLCByZXNpZGVudGlhbCwgaG90ZWwsIHJlc29ydCwgY2FzaW5vIHByb3BlcnR5LCB3YXJlaG91c2Ugc3BhY2UsIGRhdGEgY2VudGVyIFJFSVQiLCJWSUNJIFByb3BlcnRpZXMsIFByb2xvZ2lzLCBTaW1vbiBQcm9wZXJ0eSBHcm91cCwgRC5SLiBIb3J0b24sIExlbm5hciwgTWFycmlvdHQiLCJOYXRpb25hbCBIb21lYnVpbGRlcnMsIFJlZ2lvbmFsIEhvbWVidWlsZGVycywgQ2FzaW5vIFJFSVRzLCBGbGV4aWJsZSBTcGFjZSBQcm92aWRlcnMiCk1hdGVyaWFscyAmIENoZW1pY2FscywiU3BlY2lhbHR5L2NvbW1vZGl0eSBjaGVtaWNhbHMsIHBsYXN0aWNzLCBjb2F0aW5ncywgYWRoZXNpdmVzLCBpbmR1c3RyaWFsIGdhc2VzLCBjb25zdHJ1Y3Rpb24gbWF0ZXJpYWxzLCBwYXBlci9wYWNrYWdpbmcgbWF0ZXJpYWxzIiwiY2hlbWljYWwsIG1hdGVyaWFsLCBwbGFzdGljLCByZXNpbiwgcG9seW1lciwgc3BlY2lhbHR5IGNoZW1pY2FsLCBpbmR1c3RyaWFsIGdhcywgY29hdGluZywgYWRoZXNpdmUsIGFnZ3JlZ2F0ZSwgbHVtYmVyLCBwYXBlciwgcHVscCwgZmVydGlsaXplciwgcGFpbnQiLCJCQVNGLCBEb3csIExpbmRlLCBBaXIgTGlxdWlkZSwgU2hlcndpbi1XaWxsaWFtcywgTHlvbmRlbGxCYXNlbGwsIEludGVybmF0aW9uYWwgUGFwZXIiLCJTcGVjaWFsdHkgQ2hlbWljYWwgUHJvZHVjZXJzLCBJbmR1c3RyaWFsIEdhcyBDb21wZXRpdG9ycywgQWx0ZXJuYXRpdmUgTWF0ZXJpYWwgTWFudWZhY3R1cmVycywgQWdncmVnYXRlIFByb2R1Y2VycyIK",
        "competitor_types.csv": "Y29tcGV0aXRvcl90eXBlLGRlc2NyaXB0aW9uLGtleXdvcmRfc2lnbmFscyxleGFtcGxlcwpDb21wYW55LCJBIHNwZWNpZmljIG5hbWVkIGNvbXBhbnksIGNvcnBvcmF0aW9uLCBvciBsZWdhbCBlbnRpdHkuIElkZW50aWZpZWQgYnkgbGVnYWwgc3VmZml4ZXMgKEluYy4sIENvcnAuLCBMdGQuLCBldGMuKSwgd2VsbC1rbm93biBicmFuZCBuYW1lcywgb3IgYWNyb255bXMvYWJicmV2aWF0aW9ucyBvZiBrbm93biBjb21wYW5pZXMuIiwiSW5jLiwgQ29ycC4sIENvcnBvcmF0aW9uLCBMTEMsIEx0ZC4sIExpbWl0ZWQsIHBsYywgUExDLCBTLkEuLCBOLlYuLCBBRywgR21iSCwgU0UsIEwuUC4sIE95aiwgQi5WLiwgUy5wLkEuOyBBTEwtQ0FQUyBhYmJyZXZpYXRpb25zIChBTUQsIElCTSwgQk5TRik7IHdlbGwta25vd24gYnJhbmQgbmFtZXMgd2l0aG91dCBzdWZmaXggKEFwcGxlLCBCb2VpbmcsIFNhbXN1bmcpIiwiQXBwbGU7IE5WSURJQSBDb3Jwb3JhdGlvbjsgNy1FbGV2ZW4sIEluYy47IEJvZWluZzsgQVdTOyBHb29nbGUgQ2xvdWQ7IFNLIGh5bml4IEluYy47IEdlbmVudGVjaC9Sb2NoZTsgVC1Nb2JpbGU7IE9QRUMrIgpDYXRlZ29yeSwiQSBuYW1lZCBncm91cCBvciBjbGFzcyBvZiBjb21wZXRpdG9ycyBkZXNjcmliZWQgYnkgd2hhdCB0aGV5IGRvLCBub3Qgd2hvIHRoZXkgYXJlLiBUeXBpY2FsbHkgZW5kcyB3aXRoIGEgcGx1cmFsIG5vdW4gZGVzY3JpYmluZyBhIGJ1c2luZXNzIHR5cGUuIFRoZXNlIGJlY29tZSBodWIgbm9kZXMgaW4gdGhlIGdyYXBoIHZpYSBJTl9DQVRFR09SWSBlZGdlcy4iLCJwcm92aWRlcnMsIGNvbXBhbmllcywgY2FycmllcnMsIHJldGFpbGVycywgbWFudWZhY3R1cmVycywgc3VwcGxpZXJzLCBkaXN0cmlidXRvcnMsIG9wZXJhdG9ycywgaW5zdGl0dXRpb25zLCBwcm9kdWNlcnMsIGZpcm1zLCBiYW5rcywgaW5zdXJlcnMsIGJyb2tlcnMsIHBsYXRmb3JtcywgYnVzaW5lc3NlcywgbGVuZGVycywgc3RvcmVzLCBjaGFpbnMsIGRlYWxlcnMsIHV0aWxpdGllcywgZGV2ZWxvcGVycywgcHJvY2Vzc29ycywgT0VNcywgUkVJVHMsIElTUHM7IGluZHVzdHJ5LCBtYXJrZXQsIHNlY3RvciwgYnJhbmRzLCBzeXN0ZW1zLCBmYWNpbGl0aWVzLCBvcmdhbml6YXRpb25zIiwiT25saW5lIFJldGFpbGVyczsgQ2xvdWQgU2VydmljZSBQcm92aWRlcnM7IEdlbmVyaWMgTWFudWZhY3R1cmVyczsgRmludGVjaCBDb21wYW5pZXM7IFByaW1lIEJyb2tlcnM7IE1vdG9yIENhcnJpZXJzOyBJbmRlcGVuZGVudCBQb3dlciBQcm9kdWNlcnMgKElQUHMpIgpHZW5lcmljLCJBIHZhZ3VlIG9yIGRlc2NyaXB0aXZlIHJlZmVyZW5jZSB0byBjb21wZXRpdG9ycyB3aXRob3V0IG5hbWluZyBhIHNwZWNpZmljIGdyb3VwIG9yIGJ1c2luZXNzIHR5cGUuIE9mdGVuIHVzZXMgYWRqZWN0aXZlcyBvZiBzY2FsZSwgZ2VvZ3JhcGh5LCBvciBvd25lcnNoaXAgcmF0aGVyIHRoYW4gZGVzY3JpYmluZyB3aGF0IHRoZSBjb21wZXRpdG9ycyBkby4gUHJvdmlkZXMgbGl0dGxlIHN0cnVjdHVyYWwgdmFsdWUgaW4gdGhlIGdyYXBoLiIsImxhcmdlLCBzbWFsbCwgb3RoZXIsIHZhcmlvdXMsIG51bWVyb3VzLCB1bm5hbWVkLCBuZXcsIGVtZXJnaW5nLCBleGlzdGluZywgdHJhZGl0aW9uYWwsIGluZGVwZW5kZW50LCBwcml2YXRlLCBhbHRlcm5hdGl2ZSwgc3BlY2lhbGl6ZWQsIGRpdmVyc2lmaWVkLCBub24tYmFuaywgbm9uLXRyYWRpdGlvbmFsOyBvZnRlbiBsYWNrcyBhIHNwZWNpZmljIGJ1c2luZXNzLXR5cGUgbm91biIsIkxhcmdlIGdsb2JhbCBjb21wZXRpdG9yczsgU21hbGxlciBuaWNoZSBwbGF5ZXJzOyBUaGlyZC1wYXJ0eSBwcm92aWRlcnM7IFZhcmlvdXMgY29tcGV0aXRvcnM7IFVubmFtZWQgY29tcGV0aXRvcnM7IEV4aXN0aW5nIGFuZCBuZXcgZW50cmFudHMiCg==",
        "risk_categories.csv": "cmlza19jYXRlZ29yeSxkZXNjcmlwdGlvbixleGFtcGxlX3Jpc2tzLGtleXdvcmRzCkN5YmVyc2VjdXJpdHkgJiBEYXRhIFByaXZhY3ksIlRocmVhdHMgdG8gaW5mb3JtYXRpb24gc3lzdGVtcywgZGF0YSBicmVhY2hlcywgcmFuc29td2FyZSwgcHJpdmFjeSByZWd1bGF0aW9uLCBhbmQgZGF0YSBwcm90ZWN0aW9uIHJlcXVpcmVtZW50cyBpbmNsdWRpbmcgbG9jYWxpemF0aW9uIGFuZCBzb3ZlcmVpZ250eSBtYW5kYXRlcy4iLCJBSS1FbmhhbmNlZCBDeWJlcnNlY3VyaXR5IFRocmVhdHM7IEN5YmVyc2VjdXJpdHkgVGhyZWF0czsgQ3liZXJzZWN1cml0eSBhbmQgUmFuc29td2FyZTsgQ3liZXJzZWN1cml0eSBhbmQgRGF0YSBQcml2YWN5OyBEYXRhIFByaXZhY3kgYW5kIEN5YmVyc2VjdXJpdHkgUmVndWxhdGlvbjsgRXZvbHZpbmcgRGF0YSBQcml2YWN5IFJlZ3VsYXRpb25zOyBEZWVwZmFrZSBTY2hlbWVzOyBEYXRhIExvY2FsaXphdGlvbiBSZXF1aXJlbWVudHMiLCJjeWJlcixyYW5zb213YXJlLGRhdGEgYnJlYWNoLGRhdGEgcHJpdmFjeSxkYXRhIHByb3RlY3Rpb24sZ2RwcixwaGlzaGluZyxkZWVwZmFrZSxpbmZvcm1hdGlvbiBzZWN1cml0eSxkYXRhIGxvY2FsaXphdGlvbixkYXRhIHNvdmVyZWlnbnR5LGRhdGEgcmVzaWRlbmN5IgpDbGltYXRlICYgRW52aXJvbm1lbnQsIlBoeXNpY2FsIGFuZCB0cmFuc2l0aW9uIHJpc2tzIGZyb20gY2xpbWF0ZSBjaGFuZ2UsIGV4dHJlbWUgd2VhdGhlciBldmVudHMsIG5hdHVyYWwgZGlzYXN0ZXJzLCBlbnZpcm9ubWVudGFsIHJlbWVkaWF0aW9uLCBHSEcgZW1pc3Npb25zLCBhbmQgY2FyYm9uLXJlbGF0ZWQgcmVndWxhdGlvbnMuIiwiQ2xpbWF0ZSBDaGFuZ2U7IENsaW1hdGUgQ2hhbmdlIGFuZCBFbnZpcm9ubWVudGFsIFJlZ3VsYXRpb247IENsaW1hdGUgQ2hhbmdlIFRyYW5zaXRpb24gYW5kIFBoeXNpY2FsIFJpc2tzOyBDbGltYXRlIENoYW5nZSBhbmQgRXh0cmVtZSBXZWF0aGVyOyBQaHlzaWNhbCBJbXBhY3RzIG9mIENsaW1hdGUgQ2hhbmdlOyBXYXRlciBTY2FyY2l0eTsgV2lsZGZpcmUgTGlhYmlsaXR5OyBFbnZpcm9ubWVudGFsIFJlbWVkaWF0aW9uIExpYWJpbGl0aWVzIiwiY2xpbWF0ZSxnaGcsZ3JlZW5ob3VzZSxleHRyZW1lIHdlYXRoZXIsd2F0ZXIgc2NhcmNpdHksZW52aXJvbm1lbnRhbCx3aWxkZmlyZSxuYXR1cmFsIGRpc2FzdGVyLGVtaXNzaW9ucyxjYXJib24sZHJvdWdodCxmbG9vZCxodXJyaWNhbmUiClJlZ3VsYXRvcnkgJiBDb21wbGlhbmNlLCJDaGFuZ2VzIGluIGxhd3MsIHJlZ3VsYXRpb25zLCBhbmQgZW5mb3JjZW1lbnQgYWN0aW9ucyBhY3Jvc3MganVyaXNkaWN0aW9uczsgcmVndWxhdG9yeSBidXJkZW4gYW5kIGNvbXBsaWFuY2UgY29zdHM7IHBvbGl0aWNhbCBhbmQgcG9saWN5IHNoaWZ0cyBhZmZlY3RpbmcgYnVzaW5lc3Mgb3BlcmF0aW9ucy4iLCJSZWd1bGF0b3J5IGFuZCBMZWdpc2xhdGl2ZSBDaGFuZ2VzOyBSZWd1bGF0b3J5IGFuZCBFbnZpcm9ubWVudGFsIENvbXBsaWFuY2U7IFBGQVMgUmVndWxhdG9yeSBSZXN0cmljdGlvbnM7IERpZ2l0YWwgT3BlcmF0aW9uYWwgUmVzaWxpZW5jZSBBY3QgKERPUkEpOyBQb3N0LTIwMjQgVS5TLiBFbGVjdGlvbiBQb2xpY3kgU2hpZnRzOyBDYWxpZm9ybmlhIFJlZmluaW5nIE1hcmdpbiBDYXBzIChTQnggMS0yKTsgT25lIEJpZyBCZWF1dGlmdWwgQmlsbCBBY3QiLCJyZWd1bGF0LGNvbXBsaWFuY2UsbGVnaXNsYXQsbGljZW5zaW5nLGVuZm9yY2VtZW50LHBvbGljeSBzaGlmdCxleGVjdXRpdmUgb3JkZXIscG9saXRpY2FsLHJ1bGVtYWtpbmciCkFJICYgRW1lcmdpbmcgVGVjaG5vbG9neSwiUmlza3MgZnJvbSBhcnRpZmljaWFsIGludGVsbGlnZW5jZSBpbXBsZW1lbnRhdGlvbiwgZ2VuZXJhdGl2ZSBBSSwgbWFjaGluZSBsZWFybmluZywgcXVhbnR1bSBjb21wdXRpbmcsIHRlY2hub2xvZ2ljYWwgZGlzcnVwdGlvbiwgYW5kIGRpZ2l0YWwgdHJhbnNmb3JtYXRpb24uIiwiR2VuZXJhdGl2ZSBBcnRpZmljaWFsIEludGVsbGlnZW5jZTsgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UgKEFJKSBJbXBsZW1lbnRhdGlvbjsgQUkgUmVndWxhdG9yeSBDb21wbGlhbmNlOyBBcnRpZmljaWFsIEludGVsbGlnZW5jZSBSZWd1bGF0aW9uOyBRdWFudHVtIENvbXB1dGluZzsgVGVjaG5vbG9naWNhbCBEaXNydXB0aW9uOyBUZWNobm9sb2dpY2FsIE9ic29sZXNjZW5jZTsgQUkgYW5kIE1hY2hpbmUgTGVhcm5pbmcgUmVndWxhdGlvbiIsImFydGlmaWNpYWwgaW50ZWxsaWdlbmNlLGdlbmVyYXRpdmUscXVhbnR1bSxtYWNoaW5lIGxlYXJuaW5nLHRlY2hub2xvZyxkaWdpdGFsLG9ic29sZXNjZW4sYXV0b21hdGlvbiIKU3VwcGx5IENoYWluICYgT3BlcmF0aW9ucywiRGlzcnVwdGlvbnMgdG8gc3VwcGx5IGNoYWlucywgY29tcG9uZW50IHNob3J0YWdlcywgc2VtaWNvbmR1Y3RvciBzdXBwbHkgY29uc3RyYWludHMsIGxvZ2lzdGljcyBjaGFsbGVuZ2VzLCBzdXBwbGllciBjb25jZW50cmF0aW9uLCBvcGVyYXRpb25hbCBmYWlsdXJlcywgYW5kIG1hbnVmYWN0dXJpbmcgcmlza3MuIiwiU3VwcGx5IENoYWluIERpc3J1cHRpb25zOyBTdXBwbHkgQ2hhaW4gQ29uc3RyYWludHM7IFN1cHBseSBDaGFpbiBDb25jZW50cmF0aW9uOyBHbG9iYWwgU3VwcGx5IENoYWluIERpc3J1cHRpb25zOyBTdXBwbHkgQ2hhaW4gYW5kIFNlbWljb25kdWN0b3IgU2hvcnRhZ2VzOyBTZW1pY29uZHVjdG9yIENoaXAgU2hvcnRhZ2U7IFN1cHBsaWVyIENvbmNlbnRyYXRpb247IENoYXNzaXMgYW5kIENvbXBvbmVudCBTdXBwbHkgQ29uc3RyYWludHMiLCJzdXBwbHkgY2hhaW4sbG9naXN0aWNzLG1hbnVmYWN0dXJpbmcsY29tcG9uZW50IHNob3J0YWdlLHNlbWljb25kdWN0b3Isb3BlcmF0aW9uYWwgZGlzcnVwdGlvbixzdXBwbGllcix2ZW5kb3Isb3V0c291cmMsdGhpcmQtcGFydHkiCk1hY3JvZWNvbm9taWMgJiBJbmZsYXRpb24sIkJyb2FkIGVjb25vbWljIGNvbmRpdGlvbnMgaW5jbHVkaW5nIGluZmxhdGlvbiwgcmVjZXNzaW9uLCBjb3N0IHByZXNzdXJlcywgY29tbW9kaXR5IHByaWNlIHZvbGF0aWxpdHksIGlucHV0IGNvc3QgaW5jcmVhc2VzLCBhbmQgcHJpY2luZy9tYXJnaW4gY29tcHJlc3Npb24uIiwiSW5mbGF0aW9uYXJ5IFByZXNzdXJlczsgQ29tbW9kaXR5IFByaWNlIFZvbGF0aWxpdHk7IE1hY3JvZWNvbm9taWMgVm9sYXRpbGl0eTsgSW5wdXQgQ29zdCBJbmZsYXRpb247IFNvY2lhbCBJbmZsYXRpb247IFJhdyBNYXRlcmlhbCBQcmljZSBWb2xhdGlsaXR5OyBDb3N0IEluZmxhdGlvbjsgR3Jvc3MgTWFyZ2luIENvbXByZXNzaW9uOyBQcmljaW5nIGFuZCBNYXJnaW4gQ29tcHJlc3Npb24iLCJpbmZsYXRpb24sbWFjcm9lY29ub20scmVjZXNzaW9uLGVjb25vbWljIGRvd250dXJuLGNvc3QgcHJlc3N1cmUsY29tbW9kaXR5LHJhdyBtYXRlcmlhbCxpbnB1dCBjb3N0LG1hcmdpbiBjb21wcmVzcyxwcmljaW5nIHByZXNzdXJlLHByaWNlIHZvbGF0aWwsZGVmbGF0IgpGaW5hbmNpYWwgJiBDYXBpdGFsIE1hcmtldHMsIkludGVyZXN0IHJhdGUgcmlzaywgTElCT1IvU09GUiB0cmFuc2l0aW9uLCBjcmVkaXQgcmlzaywgY2FwaXRhbCBhZGVxdWFjeSwgYmFua2luZyByZWd1bGF0aW9uIChCYXNlbCBJSUkpLCBhc3NldCBpbXBhaXJtZW50LCBsaXF1aWRpdHksIGFuZCBmaW5hbmNpYWwgaW5zdHJ1bWVudCB2YWx1YXRpb24uIiwiSW50ZXJlc3QgUmF0ZSBWb2xhdGlsaXR5OyBMSUJPUiBUcmFuc2l0aW9uOyBHbG9iYWwgTWluaW11bSBUYXggKFBpbGxhciBUd28pOyBHb29kd2lsbCBJbXBhaXJtZW50OyBBc3NldCBJbXBhaXJtZW50OyBCYXNlbCBJSUkgRW5kZ2FtZSBQcm9wb3NhbDsgTmV0IEludGVyZXN0IE1hcmdpbiBDb21wcmVzc2lvbjsgRkRJQyBTcGVjaWFsIEFzc2Vzc21lbnQiLCJpbnRlcmVzdCByYXRlLGxpYm9yLHNvZnIsY3JlZGl0IHJpc2ssY2FwaXRhbCByZXF1aXJlLGltcGFpcm1lbnQsZ29vZHdpbGwsbGlxdWlkaXR5LGJhc2VsLGZkaWMsYmFua2luZyxnLXNpYixuZXQgaW50ZXJlc3QgbWFyZ2luLGZlZSBjb21wcmVzcyIKRm9yZWlnbiBDdXJyZW5jeSAmIEV4Y2hhbmdlLCJFeHBvc3VyZSB0byBmb3JlaWduIGN1cnJlbmN5IGZsdWN0dWF0aW9ucywgZXhjaGFuZ2UgcmF0ZSB2b2xhdGlsaXR5LCB0cmFuc2xhdGlvbiBpbXBhY3RzLCBhbmQgaGVkZ2luZyByaXNrcy4iLCJGb3JlaWduIEN1cnJlbmN5IFZvbGF0aWxpdHk7IEZvcmVpZ24gQ3VycmVuY3kgRXhjaGFuZ2UgUmF0ZSBGbHVjdHVhdGlvbnM7IEZvcmVpZ24gRXhjaGFuZ2UgVm9sYXRpbGl0eTsgRm9yZWlnbiBDdXJyZW5jeSBUcmFuc2xhdGlvbjsgQXJnZW50aW5lIFBlc28gRGV2YWx1YXRpb247IEhlZGdpbmcgSW5lZmZlY3RpdmVuZXNzIGFuZCBNYXJrZXQgVm9sYXRpbGl0eSIsImN1cnJlbmN5LGZvcmVpZ24gZXhjaGFuZ2UsZm9yZXgsZnggLHRyYW5zbGF0aW9uLGhlZGdpbmcscGVzbyxleGNoYW5nZSByYXRlIgpHZW9wb2xpdGljYWwgJiBUcmFkZSwiSW50ZXJuYXRpb25hbCBwb2xpdGljYWwgaW5zdGFiaWxpdHksIGFybWVkIGNvbmZsaWN0cywgdHJhZGUgd2FycywgdGFyaWZmcywgc2FuY3Rpb25zLCBleHBvcnQgY29udHJvbHMsIGFuZCBjcm9zcy1ib3JkZXIgcmVzdHJpY3Rpb25zLiIsIkdlb3BvbGl0aWNhbCBJbnN0YWJpbGl0eTsgVHJhZGUgUG9saWN5IGFuZCBUYXJpZmZzOyBSdXNzaWEtVWtyYWluZSBDb25mbGljdDsgR2VvcG9saXRpY2FsIENvbmZsaWN0czsgUmVkIFNlYSBTaGlwcGluZyBEaXNydXB0aW9uczsgVS5TLiBFeHBvcnQgQ29udHJvbHMgYW5kIEVudGl0eSBMaXN0IFJlc3RyaWN0aW9uczsgTWlkZGxlIEVhc3QgQ29uZmxpY3Q7IFVTTUNBIFJlbmV3YWwiLCJnZW9wb2xpdCx0YXJpZmYsdHJhZGUsc2FuY3Rpb24sZXhwb3J0IGNvbnRyb2wscnVzc2lhLHVrcmFpbmUsY2hpbmEsY29uZmxpY3QsZW1iYXJnbyxwcm90ZWN0aW9uaXNtIgpUYXggJiBGaXNjYWwgUG9saWN5LCJDaGFuZ2VzIGluIHRheCBsYXdzLCBnbG9iYWwgbWluaW11bSB0YXggaW5pdGlhdGl2ZXMsIGZpc2NhbCBwb2xpY3kgc2hpZnRzLCB0cmFuc2ZlciBwcmljaW5nLCBhbmQgdGF4IGRpc3B1dGUgcmlza3MuIiwiR2xvYmFsIE1pbmltdW0gVGF4IChQaWxsYXIgVHdvKTsgT0VDRCBQaWxsYXIgVHdvIEdsb2JhbCBNaW5pbXVtIFRheDsgQ29ycG9yYXRlIEFsdGVybmF0aXZlIE1pbmltdW0gVGF4IChDQU1UKTsgSVJTIFRheCBEaXNwdXRlOyBTaGFyZSBSZXB1cmNoYXNlIEV4Y2lzZSBUYXg7IFRyYW5zZmVyIFByaWNpbmcgQXVkaXRzOyBTZWN0aW9uIDE3NCBSJkQgQW1vcnRpemF0aW9uIiwidGF4LHBpbGxhciB0d28sbWluaW11bSB0YXgsY2FtdCxpcnMsZmlzY2FsLHRyYW5zZmVyIHByaWNpbmcsZXhjaXNlIgpMYWJvciAmIEh1bWFuIENhcGl0YWwsIkxhYm9yIG1hcmtldCBjaGFsbGVuZ2VzIGluY2x1ZGluZyBzaG9ydGFnZXMsIHVuaW9uaXphdGlvbiwgd2FnZSBwcmVzc3VyZXMsIHRhbGVudCByZXRlbnRpb24sIHdvcmtlciByZWNsYXNzaWZpY2F0aW9uLCByZW1vdGUvaHlicmlkIHdvcmsgaW1wYWN0cywgYW5kIHdvcmtmb3JjZSBtYW5hZ2VtZW50LiIsIkxhYm9yIFVuaW9uaXphdGlvbjsgTGFib3IgU2hvcnRhZ2VzOyBMYWJvciBTaG9ydGFnZXMgYW5kIFdhZ2UgSW5mbGF0aW9uOyBIdW1hbiBDYXBpdGFsIENvbXBldGl0aW9uOyBXb3JrZXIgUmVjbGFzc2lmaWNhdGlvbjsgSHlicmlkIFdvcmsgTW9kZWwgQ2hhbGxlbmdlczsgRHJpdmVyIFJlY2xhc3NpZmljYXRpb247IFBpbG90IFN0YWZmaW5nIENvbnN0cmFpbnRzIiwibGFib3Isd29ya2ZvcmNlLHRhbGVudCx1bmlvbix3YWdlLHdvcmtlcixodW1hbiBjYXBpdGFsLGVtcGxveWVlLHJlbW90ZSB3b3JrLGh5YnJpZCB3b3JrLHN0YWZmaW5nLGF0dHJpdGlvbixyZWNydWl0bWVudCIKTGVnYWwgJiBMaXRpZ2F0aW9uLCJMYXdzdWl0cywgaW50ZWxsZWN0dWFsIHByb3BlcnR5IGRpc3B1dGVzLCBjbGFzcyBhY3Rpb25zLCBwcm9kdWN0IGxpYWJpbGl0eSwgcmVndWxhdG9yeSBlbmZvcmNlbWVudCBwcm9jZWVkaW5ncywgYW50aXRydXN0IHNjcnV0aW55LCBhbmQgc2V0dGxlbWVudCByaXNrcy4iLCJJbnRlbGxlY3R1YWwgUHJvcGVydHkgTGl0aWdhdGlvbjsgTWFzcyBUb3J0IENsYWltczsgQW50aXRydXN0IFNjcnV0aW55OyBOdWNsZWFyIFZlcmRpY3RzOyBXaGlzdGxlYmxvd2VyIGFuZCBRdWkgVGFtIEFjdGlvbnM7IFNFQyBJbnZlc3RpZ2F0aW9uOyBGVEMgQW50aXRydXN0IENoYWxsZW5nZTsgRE9KIEZhbHNlIENsYWltcyBBY3QgSW52ZXN0aWdhdGlvbiIsImxpdGlnYXRpb24sbGF3c3VpdCxsZWdhbCxpbnRlbGxlY3R1YWwgcHJvcGVydHkscGF0ZW50LGxpYWJpbGl0eSxhbnRpdHJ1c3QsY2xhc3MgYWN0aW9uLHNldHRsZW1lbnQsaW5kZW1uaWYiCkhlYWx0aGNhcmUgJiBQaGFybWEsIkhlYWx0aGNhcmUgcG9saWN5LCBkcnVnIHByaWNpbmcgcmVndWxhdGlvbiwgY2xpbmljYWwgdHJpYWwgcmlza3MsIG1lZGljYWwgY29zdCB0cmVuZHMsIEZEQSBhY3Rpb25zLCByZWltYnVyc2VtZW50IGNoYW5nZXMsIGJpb3NpbWlsYXIgY29tcGV0aXRpb24sIGFuZCBHTFAtMSBkcnVnIGltcGFjdHMuIiwiTWVkaWNhaWQgUmVkZXRlcm1pbmF0aW9uczsgSW5mbGF0aW9uIFJlZHVjdGlvbiBBY3QgKElSQSkgUHJpY2UgTmVnb3RpYXRpb25zOyBFWUxFQSBCaW9zaW1pbGFyIENvbXBldGl0aW9uOyBDbGluaWNhbCBUcmlhbCBGYWlsdXJlOyBHTFAtMSBSZWNlcHRvciBBZ29uaXN0czsgQ01TIFN0YXIgUmF0aW5ncyBEZWNsaW5lOyBNZWRpY2FsIENvc3QgVHJlbmQgVm9sYXRpbGl0eTsgTG9zcyBvZiBFeGNsdXNpdml0eSIsIm1lZGljYWlkLG1lZGljYXJlLGRydWcgcHJpYyxiaW9zaW1pbGFyLHJlaW1idXJzZW1lbnQscGhhcm1hLGhlYWx0aGNhcmUsZmRhLGNsaW5pY2FsLGdscC0xLHdlaWdodCBsb3NzLG1lZGljYWwgY29zdCxzdGFyIHJhdGluZywzNDBiLGNtcyIKRVNHICYgU3VzdGFpbmFiaWxpdHksIkVudmlyb25tZW50YWwsIHNvY2lhbCwgYW5kIGdvdmVybmFuY2Ugc2NydXRpbnkgZnJvbSBpbnZlc3RvcnMgYW5kIHJlZ3VsYXRvcnM7IGFudGktRVNHIGJhY2tsYXNoOyBzdXN0YWluYWJpbGl0eSBtYW5kYXRlczsgZ3JlZW53YXNoaW5nIHJpc2s7IGFuZCBzdGFrZWhvbGRlciBleHBlY3RhdGlvbnMuIiwiRVNHIFNjcnV0aW55OyBBbnRpLUVTRyBTZW50aW1lbnQ7IEVTRyBhbmQgU3VzdGFpbmFiaWxpdHkgU2NydXRpbnk7IEVTRyBSZWd1bGF0b3J5IENvbXBsaWFuY2U7IEVTRy1Ecml2ZW4gQ2FwaXRhbCBDb25zdHJhaW50czsgQ29ycG9yYXRlIENpdGl6ZW5zaGlwIGFuZCBHcmVlbndhc2hpbmc7IFN1c3RhaW5hYmxlIEZpbmFuY2UgRGlzY2xvc3VyZSBhbmQgR3JlZW53YXNoaW5nIiwiZXNnLHN1c3RhaW5hYmlsaXR5LHN0YWtlaG9sZGVyLGdyZWVud2FzaCxyZXNwb25zaWJsZSBpbnZlc3QsZGl2ZXN0bWVudCIKQ29tcGV0aXRpb24gJiBNYXJrZXQgUG9zaXRpb24sIkNvbXBldGl0aXZlIHByZXNzdXJlcywgbWFya2V0IHNoYXJlIGVyb3Npb24sIGluZHVzdHJ5IGRpc3J1cHRpb24gZnJvbSBuZXcgZW50cmFudHMsIG1hcmtldCBzYXR1cmF0aW9uLCBpbmR1c3RyeSBjb25zb2xpZGF0aW9uLCBhbmQgc2hpZnRzIGluIGJ1c2luZXNzIG1vZGVscy4iLCJDdXN0b21lciBDb25jZW50cmF0aW9uOyBJbmR1c3RyeSBDb25zb2xpZGF0aW9uOyBJbnZlbnRvcnkgU2hyaW5rOyBOb24tdHJhZGl0aW9uYWwgQ29tcGV0aXRvcnM7IExvdyBCYXJyaWVycyB0byBFbnRyeTsgTWFya2V0IFNhdHVyYXRpb247IFNtYXJ0cGhvbmUgTWFya2V0IFNhdHVyYXRpb247IEZpbnRlY2ggYW5kIERpc2ludGVybWVkaWF0aW9uIiwiY29tcGV0aXRpb24sY29tcGV0aXRpdmUsbWFya2V0IHNoYXJlLGRpc3J1cHQsY29uc29saWRhdCxuZXcgZW50cmFudCxwcml2YXRlIGxhYmVsLG1hcmtldCBzYXR1cmF0aW9uLGRpc2ludGVybWVkaWF0aW9uLGZyYWdtZW50YXRpb24iCkNvcnBvcmF0ZSBTdHJhdGVneSAmIEV4ZWN1dGlvbiwiUmlza3MgZnJvbSBNJkEgaW50ZWdyYXRpb24sIGNvcnBvcmF0ZSByZXN0cnVjdHVyaW5nLCBzcGluLW9mZnMsIEVSUCBpbXBsZW1lbnRhdGlvbnMsIGFjY291bnRpbmcgc3RhbmRhcmQgY2hhbmdlcywgc3RyYXRlZ2ljIGluaXRpYXRpdmUgZXhlY3V0aW9uLCBhbmQgb3BlcmF0aW9uYWwgdHJhbnNmb3JtYXRpb24uIiwiRVJQIFN5c3RlbSBJbXBsZW1lbnRhdGlvbjsgR29vZHdpbGwgSW1wYWlybWVudDsgQnVzaW5lc3MgTW9kZWwgVHJhbnNmb3JtYXRpb247IFJlc3RydWN0dXJpbmcgYW5kIEV4ZWN1dGlvbiBSaXNrOyBTcGluLW9mZiBFeGVjdXRpb24gVW5jZXJ0YWludHk7IFJldmVudWUgUmVjb2duaXRpb24gQ29tcGxleGl0eTsgTERUSSBBY2NvdW50aW5nIENoYW5nZXM7IE1hdGVyaWFsIFdlYWtuZXNzIGluIEludGVybmFsIENvbnRyb2xzIiwiYWNxdWksbWVyZ2VyLGRpdmVzdGl0dXJlLGludGVncmF0aW9uLGpvaW50IHZlbnR1cmUscmVzdHJ1Y3R1cixzcGluLW9mZixlcnAsYWNjb3VudGluZyxyZXZlbnVlIHJlY29nbml0aW9uLG1hdGVyaWFsIHdlYWtuZXNzLHNlcGFyYXRpb24sdHJhbnNmb3JtYXRpb24iCkN1c3RvbWVyICYgRGVtYW5kLCJDdXN0b21lciBjb25jZW50cmF0aW9uLCBzaGlmdHMgaW4gY29uc3VtZXIgYmVoYXZpb3IsIGRlbWFuZCB2b2xhdGlsaXR5LCBjaGFuZ2luZyBjb25zdW1wdGlvbiBwYXR0ZXJucywgYW5kIGNoYW5uZWwgZGlzcnVwdGlvbi4iLCJDdXN0b21lciBDb25jZW50cmF0aW9uOyBEZWNsaW5pbmcgSG9tZSBBZmZvcmRhYmlsaXR5OyBDb3JkLUN1dHRpbmcgYW5kIE1WUEQgU3Vic2NyaWJlciBEZWNsaW5lOyBMaW5lYXIgVGVsZXZpc2lvbiBEZWNsaW5lOyBBdWRpZW5jZSBGcmFnbWVudGF0aW9uOyBEaXNjcmV0aW9uYXJ5IFNwZW5kaW5nIFZvbGF0aWxpdHk7IEUtY29tbWVyY2UgUHJpY2UgRGVmbGF0aW9uOyBDb25uZWN0ZWQgRml0bmVzcyBNYXJrZXQgRGVjbGluZSIsImN1c3RvbWVyLGNsaWVudCxkZW1hbmQsY29uc3VtZXIsc3Vic2NyaWJlcixjaHVybixjb3JkLWN1dCxhdWRpZW5jZSxkaXNjcmV0aW9uYXJ5IHNwZW5kaW5nLGJyYW5kLGxveWFsdHkiCg==",
        "market_geographic_regions.csv": "Z2VvZ3JhcGhpY19yZWdpb24sZGVzY3JpcHRpb24sa2V5d29yZHMsZXhhbXBsZXMKR2xvYmFsLCJXb3JsZHdpZGUgb3IgbXVsdGktY29udGluZW50YWwgb3BlcmF0aW9ucywgbm8gc3BlY2lmaWMgcmVnaW9uIiwiaW50ZXJuYXRpb25hbCwgZ2xvYmFsLCB3b3JsZHdpZGUsIG92ZXJzZWFzLCBtdWx0aW5hdGlvbmFsLCBjcm9zcy1ib3JkZXIsIGZvcmVpZ24iLCJJbnRlcm5hdGlvbmFsIE1hcmtldHMsIEdsb2JhbCBPcGVyYXRpb25zLCBXb3JsZHdpZGUgRXhwYW5zaW9uIgpOb3J0aCBBbWVyaWNhLCJVbml0ZWQgU3RhdGVzLCBDYW5hZGEsIE1leGljbyIsInVuaXRlZCBzdGF0ZXMsIHUucy4sIHVzYSwgYW1lcmljYSwgY2FuYWRhLCBtZXhpY28sIG5vcnRoIGFtZXJpY2EsIGRvbWVzdGljIiwiVW5pdGVkIFN0YXRlcywgTm9ydGggQW1lcmljYSwgUGVybWlhbiBCYXNpbiwgRGVsYXdhcmUgQmFzaW4sIE1leGljbyIKRXVyb3BlLCJFdXJvcGVhbiBjb3VudHJpZXMgYW5kIEVVIiwiZXVyb3BlLCBldSwgdWssIHVuaXRlZCBraW5nZG9tLCBnZXJtYW55LCBmcmFuY2UsIGl0YWx5LCBzcGFpbiwgbmV0aGVybGFuZHMsIGlyZWxhbmQsIHN3ZWRlbiwgbm9yd2F5LCBzd2l0emVybGFuZCwgcG9sYW5kLCBiZWxnaXVtLCBhdXN0cmlhLCBkZW5tYXJrLCBmaW5sYW5kLCBjemVjaCwgcG9ydHVnYWwsIHJvbWFuaWEsIGdyZWVjZSwgaHVuZ2FyeSwgdHVya2V5IiwiRXVyb3BlLCBVbml0ZWQgS2luZ2RvbSwgV2VzdGVybiBFdXJvcGUsIEV1cm9wZWFuIFVuaW9uIgpBc2lhIFBhY2lmaWMsIkVhc3QgQXNpYSwgU291dGhlYXN0IEFzaWEsIFNvdXRoIEFzaWEsIE9jZWFuaWEiLCJhc2lhLCBjaGluYSwgamFwYW4sIGluZGlhLCBzb3V0aCBrb3JlYSwgdGFpd2FuLCBzaW5nYXBvcmUsIGF1c3RyYWxpYSwgbmV3IHplYWxhbmQsIGluZG9uZXNpYSwgbWFsYXlzaWEsIHRoYWlsYW5kLCB2aWV0bmFtLCBwaGlsaXBwaW5lcywgaG9uZyBrb25nLCBhcGFjLCBhc2lhIHBhY2lmaWMsIGFzaWEtcGFjaWZpYyIsIkNoaW5hLCBJbmRpYSwgSmFwYW4sIEFzaWEgUGFjaWZpYywgU291dGhlYXN0IEFzaWEsIEF1c3RyYWxpYSIKTGF0aW4gQW1lcmljYSwiQ2VudHJhbCBhbmQgU291dGggQW1lcmljYSwgQ2FyaWJiZWFuIiwibGF0aW4gYW1lcmljYSwgc291dGggYW1lcmljYSwgY2VudHJhbCBhbWVyaWNhLCBicmF6aWwsIGFyZ2VudGluYSwgY29sb21iaWEsIGNoaWxlLCBwZXJ1LCBjYXJpYmJlYW4sIG1leGljbyIsIkJyYXppbCwgTGF0aW4gQW1lcmljYSwgU291dGggQW1lcmljYSwgQXJnZW50aW5hIgpNaWRkbGUgRWFzdCAmIEFmcmljYSwiTWlkZGxlIEVhc3QsIE5vcnRoIEFmcmljYSwgU3ViLVNhaGFyYW4gQWZyaWNhIiwibWlkZGxlIGVhc3QsIGFmcmljYSwgc2F1ZGkgYXJhYmlhLCB1YWUsIHFhdGFyLCBvbWFuLCBpc3JhZWwsIGVneXB0LCBuaWdlcmlhLCBzb3V0aCBhZnJpY2EsIG1lbmEsIGd1bGYiLCJNaWRkbGUgRWFzdCwgQWZyaWNhLCBTYXVkaSBBcmFiaWEsIFVBRSwgU3ViLVNhaGFyYW4gQWZyaWNhIgpSdXNzaWEgJiBDSVMsIlJ1c3NpYSwgQmVsYXJ1cywgVWtyYWluZSwgZm9ybWVyIFNvdmlldCBzdGF0ZXMiLCJydXNzaWEsIGJlbGFydXMsIHVrcmFpbmUsIGNpcywgcnVzc2lhbiwgY3JpbWVhIiwiUnVzc2lhLCBSdXNzaWEgYW5kIEJlbGFydXMsIFVrcmFpbmUiCg==",
        "market_product_categories.csv": "cHJvZHVjdF9jYXRlZ29yeSxkZXNjcmlwdGlvbixrZXl3b3JkcyxleGFtcGxlcwpDbG91ZCAmIFNvZnR3YXJlLCJDbG91ZCBjb21wdXRpbmcsIFNhYVMsIGVudGVycHJpc2Ugc29mdHdhcmUsIEFJL01MIHBsYXRmb3JtcyIsImNsb3VkLCBzb2Z0d2FyZSwgU2FhUywgcGxhdGZvcm0sIEFJLCBhcnRpZmljaWFsIGludGVsbGlnZW5jZSwgbWFjaGluZSBsZWFybmluZywgZ2VuZXJhdGl2ZSBBSSwgZGF0YSBjZW50ZXIsIGN5YmVyc2VjdXJpdHksIGRpZ2l0YWwgdHJhbnNmb3JtYXRpb24iLCJDbG91ZCBDb21wdXRpbmcsIEFydGlmaWNpYWwgSW50ZWxsaWdlbmNlLCBTYWFTIFBsYXRmb3JtcywgRGF0YSBDZW50ZXJzLCBHZW5lcmF0aXZlIEFJIgpSZW5ld2FibGUgRW5lcmd5LCJTb2xhciwgd2luZCwgaHlkcm9nZW4sIGJpb2Z1ZWxzLCBjbGVhbiBlbmVyZ3kiLCJyZW5ld2FibGUsIHNvbGFyLCB3aW5kLCBoeWRyb2dlbiwgYmlvZnVlbCwgY2xlYW4gZW5lcmd5LCBncmVlbiBlbmVyZ3ksIHJlbmV3YWJsZSBkaWVzZWwsIHJlbmV3YWJsZSBuYXR1cmFsIGdhcywgUk5HLCBiaW9kaWVzZWwiLCJSZW5ld2FibGUgRW5lcmd5LCBTb2xhciwgV2luZCBFbmVyZ3ksIFJlbmV3YWJsZSBEaWVzZWwsIFJlbmV3YWJsZSBOYXR1cmFsIEdhcyAoUk5HKSIKRWxlY3RyaWMgVmVoaWNsZXMgJiBNb2JpbGl0eSwiRVZzLCBjaGFyZ2luZyBpbmZyYXN0cnVjdHVyZSwgYmF0dGVyaWVzLCBhdXRvbm9tb3VzIGRyaXZpbmciLCJlbGVjdHJpYyB2ZWhpY2xlLCBFViwgY2hhcmdpbmcsIGJhdHRlcnksIGVsZWN0cmlmaWNhdGlvbiwgYXV0b25vbW91cywgc2VsZi1kcml2aW5nLCBjb25uZWN0ZWQgdmVoaWNsZSwgbW9iaWxpdHksIHJpZGUtc2hhcmluZyIsIkVsZWN0cmljIFZlaGljbGUgKEVWKSBJbmZyYXN0cnVjdHVyZSwgQmF0dGVyeSBFbGVjdHJpYyBWZWhpY2xlcywgRVYgQ2hhcmdpbmcsIEF1dG9ub21vdXMgVmVoaWNsZXMiCk9pbCAmIEdhcywiVXBzdHJlYW0vbWlkc3RyZWFtL2Rvd25zdHJlYW0gcGV0cm9sZXVtLCBMTkcsIHBpcGVsaW5lcyIsIm9pbCwgZ2FzLCBwZXRyb2xldW0sIExORywgcGlwZWxpbmUsIHJlZmluaW5nLCBkcmlsbGluZywgZXhwbG9yYXRpb24sIHNoYWxlLCBiYXNpbiwgY3J1ZGUsIHVwc3RyZWFtLCBtaWRzdHJlYW0sIGRvd25zdHJlYW0sIG5hdHVyYWwgZ2FzIiwiUGVybWlhbiBCYXNpbiwgRGVsYXdhcmUgQmFzaW4sIExORywgTmF0dXJhbCBHYXMsIENydWRlIE9pbCIKSGVhbHRoY2FyZSBTZXJ2aWNlcywiUGhhcm1hY2V1dGljYWxzLCBtZWRpY2FsIGRldmljZXMsIGhlYWx0aCBzZXJ2aWNlcywgZGlhZ25vc3RpY3MiLCJwaGFybWEsIGJpb3RlY2gsIG1lZGljYWwsIGJpb3NpbWlsYXIsIG1lZGljYXJlLCBtZWRpY2FpZCwgaG9zcGl0YWwsIGRpYWdub3N0aWMsIHRoZXJhcGV1dGljLCBjbGluaWNhbCwgZHJ1Zywgb25jb2xvZ3ksIGltbXVub2xvZ3kiLCJCaW9zaW1pbGFycywgTWVkaWNhcmUgQWR2YW50YWdlLCBTcGVjaWFsdHkgUGhhcm1hY3ksIE9uY29sb2d5LCBHZW5lIFRoZXJhcHkiCkZpbmFuY2lhbCBQcm9kdWN0cywiQmFua2luZyBwcm9kdWN0cywgaW5zdXJhbmNlLCBwYXltZW50cywgZmludGVjaCIsImJhbmtpbmcsIGluc3VyYW5jZSwgcGF5bWVudCwgZmludGVjaCwgbGVuZGluZywgbW9ydGdhZ2UsIHdlYWx0aCBtYW5hZ2VtZW50LCBhc3NldCBtYW5hZ2VtZW50LCB0cmFkaW5nLCBjcmVkaXQsIGRpZ2l0YWwgYmFua2luZyIsIkRpZ2l0YWwgUGF5bWVudHMsIEluc3VyYW5jZSwgQXNzZXQgTWFuYWdlbWVudCwgRGlnaXRhbCBCYW5raW5nLCBCdXkgTm93IFBheSBMYXRlciIKRS1jb21tZXJjZSAmIERpZ2l0YWwsIk9ubGluZSByZXRhaWwsIGRpZ2l0YWwgY29tbWVyY2UsIG1hcmtldHBsYWNlcyIsImUtY29tbWVyY2UsIGVjb21tZXJjZSwgZGlnaXRhbCBjb21tZXJjZSwgb25saW5lLCBtYXJrZXRwbGFjZSwgZGlyZWN0LXRvLWNvbnN1bWVyLCBEVEMsIG9tbmljaGFubmVsLCBsYXN0LW1pbGUiLCJFLWNvbW1lcmNlLCBEaWdpdGFsIENvbW1lcmNlLCBPbmxpbmUgTWFya2V0cGxhY2UsIERpcmVjdC10by1Db25zdW1lciIKVGVsZWNvbW11bmljYXRpb25zLCI1RywgYnJvYWRiYW5kLCB3aXJlbGVzcywgZmliZXIsIHNhdGVsbGl0ZSIsIjVHLCBicm9hZGJhbmQsIHdpcmVsZXNzLCBmaWJlciwgc2F0ZWxsaXRlLCB0ZWxlY29tLCBzcGVjdHJ1bSwgbmV0d29yaywgY29ubmVjdGl2aXR5LCBJb1QiLCI1RywgQnJvYWRiYW5kLCBGaWJlciBPcHRpYywgU2F0ZWxsaXRlIENvbW11bmljYXRpb25zLCBJb1QiClBvd2VyICYgVXRpbGl0aWVzLCJFbGVjdHJpY2l0eSBnZW5lcmF0aW9uLCB0cmFuc21pc3Npb24sIGRpc3RyaWJ1dGlvbiwgZ3JpZCIsInBvd2VyLCBlbGVjdHJpY2l0eSwgZ2VuZXJhdGlvbiwgdHJhbnNtaXNzaW9uLCBkaXN0cmlidXRpb24sIGdyaWQsIHV0aWxpdHksIG51Y2xlYXIsIGNvYWwsIGdhcy1maXJlZCwgcGVha2luZyIsIkNvYWwtZmlyZWQgR2VuZXJhdGlvbiwgTnVjbGVhciBQb3dlciwgRGlzdHJpYnV0ZWQgR2VuZXJhdGlvbiwgR3JpZCBNb2Rlcm5pemF0aW9uIgpGb29kICYgQWdyaWN1bHR1cmUsIkZvb2QgcHJvZHVjdGlvbiwgYWdyaWN1bHR1cmUsIGNyb3AgcHJvdGVjdGlvbiwgYW5pbWFsIGhlYWx0aCIsImZvb2QsIGFncmljdWx0dXJlLCBjcm9wLCBhbmltYWwsIGZlZWQsIGZlcnRpbGl6ZXIsIHNlZWQsIGRhaXJ5LCBtZWF0LCBwb3VsdHJ5LCBncmFpbiwgb3JnYW5pYyIsIk9yZ2FuaWMgRm9vZCwgUGxhbnQtYmFzZWQgUHJvdGVpbiwgQ3JvcCBQcm90ZWN0aW9uLCBBbmltYWwgSGVhbHRoIgpDb25zdHJ1Y3Rpb24gJiBJbmZyYXN0cnVjdHVyZSwiQnVpbGRpbmcsIHJlYWwgZXN0YXRlIGRldmVsb3BtZW50LCBpbmZyYXN0cnVjdHVyZSBwcm9qZWN0cyIsImNvbnN0cnVjdGlvbiwgaW5mcmFzdHJ1Y3R1cmUsIGhvdXNpbmcsIHJlc2lkZW50aWFsLCBjb21tZXJjaWFsLCBpbmR1c3RyaWFsIHNwYWNlLCBkYXRhIGNlbnRlciBjb25zdHJ1Y3Rpb24sIGhpZ2h3YXkiLCJSZXNpZGVudGlhbCBDb25zdHJ1Y3Rpb24sIERhdGEgQ2VudGVyIENvbnN0cnVjdGlvbiwgSW5mcmFzdHJ1Y3R1cmUiCkRlZmVuc2UgJiBTcGFjZSwiTWlsaXRhcnkgc3lzdGVtcywgc3BhY2UgdGVjaG5vbG9neSwgZ292ZXJubWVudCBzZXJ2aWNlcyIsImRlZmVuc2UsIG1pbGl0YXJ5LCBzcGFjZSwgc2F0ZWxsaXRlLCBtaXNzaWxlLCBpbnRlbGxpZ2VuY2UsIGdvdmVybm1lbnQsIGNsYXNzaWZpZWQsIGh5cGVyc29uaWMiLCJIeXBlcnNvbmljIFN5c3RlbXMsIFNwYWNlIExhdW5jaCwgTWlzc2lsZSBEZWZlbnNlLCBJbnRlbGxpZ2VuY2UgU2VydmljZXMiCg==",
    }
    restored = 0
    for name, b64 in embedded.items():
        p = data_dir / name
        if not p.exists():
            p.write_bytes(base64.b64decode(b64))
            restored += 1
    if restored:
        print(f"Restored {restored} missing taxonomy file(s) under {data_dir}")

ensure_entity_norm_taxonomy_files()

def run_step(cmd):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd="python_pipeline", env=py_env, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr)
        raise RuntimeError(f"Step failed ({p.returncode}): {' '.join(cmd)}")

def parquet_rows(path):
    p = Path(path)
    if not p.exists():
        return 0
    return len(pd.read_parquet(p))

print("Running Python Parquet Pipeline (Transformation & Normalization)...")
run_step(["uv", "run", "python", "transform.py"])

parquet_dir = Path("python_pipeline") / "output" / PIPELINE_MODEL_NAME / "parquet"
n_competitors = parquet_rows(parquet_dir / "nodes_competitor.parquet")
n_risks = parquet_rows(parquet_dir / "nodes_risk.parquet")
n_markets = parquet_rows(parquet_dir / "nodes_market.parquet")
print(f"Detected rows -> competitors: {n_competitors}, risks: {n_risks}, markets: {n_markets}")

if n_competitors > 0:
    run_step(["uv", "run", "python", "entity_normalization/resolve_competitors.py", "--model", PIPELINE_MODEL_NAME])
    run_step([
        "uv", "run", "python", "entity_normalization/categorize_competitor_markets.py",
        "--model", PIPELINE_MODEL_NAME,
        "--parquet-dir", f"output/{PIPELINE_MODEL_NAME}/parquet"
    ])
else:
    print("Skipping competitor normalization (0 competitor rows)")

if n_risks > 0:
    run_step(["uv", "run", "python", "entity_normalization/categorize_risks.py", "--model", PIPELINE_MODEL_NAME])
else:
    print("Skipping risk categorization (0 risk rows)")

if n_markets > 0:
    run_step(["uv", "run", "python", "entity_normalization/categorize_markets.py", "--model", PIPELINE_MODEL_NAME])
else:
    print("Skipping market categorization (0 market rows)")


## 7. Load Parquet files to BigQuery

Upload the generated Parquet files to Google Cloud Storage and load them into native BigQuery tables.

In [ ]:
PIPELINE_MODEL_NAME = globals().get("PIPELINE_MODEL_NAME", GEMINI_MODEL)
PARQUET_LOCAL_DIR = f"python_pipeline/output/{PIPELINE_MODEL_NAME}/parquet"
gcs_parquet_path = f"{GCS_BUCKET}/parquets/{PIPELINE_MODEL_NAME}/parquet"
print(f"Uploading {PARQUET_LOCAL_DIR} to {gcs_parquet_path} ...")

# Use gsutil rsync to copy the folder up to GCS
rsync_cmd = ["gsutil", "-m", "rsync", "-r", PARQUET_LOCAL_DIR, gcs_parquet_path]
subprocess.run(rsync_cmd, check=True)
print("Upload complete!")


In [ ]:
import os
import pandas as pd

tables_to_load = {
    # -- Nodes (Base) --
    "nodes_company": "nodes_company.parquet",
    "nodes_document": "nodes_document.parquet",
    "nodes_section": "nodes_section.parquet",
    "nodes_reference": "nodes_reference.parquet",
    "nodes_opportunity": "nodes_opportunity.parquet",
    "nodes_competitor": "nodes_competitor.parquet",
    # -- Nodes (Categorized / Normalized Replacements) --
    "nodes_market": "nodes_market_categorized.parquet",
    "nodes_risk": "nodes_risk_categorized.parquet",
    # -- Nodes (Normalization Taxonomy) --
    "nodes_normalized_competitor": "nodes_normalized_competitor.parquet",
    "nodes_geographic_region": "nodes_geographic_region.parquet",
    "nodes_market_category": "nodes_market_category.parquet",
    "nodes_risk_category": "nodes_risk_category.parquet",

    # -- Edges (Base) --
    "edges_filed": "edges_filed.parquet",
    "edges_doc_contains_section": "edges_doc_contains_section.parquet",
    "edges_section_contains_ref": "edges_section_contains_ref.parquet",
    "edges_entering": "edges_entering.parquet",
    "edges_exiting": "edges_exiting.parquet",
    "edges_expanding": "edges_expanding.parquet",
    "edges_faces_risk": "edges_faces_risk.parquet",
    "edges_pursuing": "edges_pursuing.parquet",
    "edges_competes": "edges_competes.parquet",
    "edges_market_has_reference": "edges_market_has_reference.parquet",
    "edges_risk_has_reference": "edges_risk_has_reference.parquet",
    "edges_opportunity_has_reference": "edges_opportunity_has_reference.parquet",
    "edges_competitor_has_reference": "edges_competitor_has_reference.parquet",

    # -- Edges (Normalization Links) --
    "edges_instance_of": "edges_instance_of.parquet",
    "edges_subsidiary_of": "edges_subsidiary_of.parquet",
    "edges_has_risk_category": "edges_has_risk_category.parquet",
    "edges_in_region": "edges_in_region.parquet",
    "edges_in_product_category": "edges_in_product_category.parquet",
    "edges_in_market_category": "edges_in_market_category.parquet"
}

loaded_tables = []
skipped_tables = []

for bq_table_name, parquet_file in tables_to_load.items():
    uri = f"{gcs_parquet_path}/{parquet_file}"
    local_parquet = f"{PARQUET_LOCAL_DIR}/{parquet_file}"

    # Skip missing files in GCS
    exists = subprocess.run(["gsutil", "ls", uri], capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"Skipping {bq_table_name}: missing {uri}")
        skipped_tables.append((bq_table_name, parquet_file))
        continue

    # Skip parquet files with zero columns (invalid for BigQuery LOAD DATA)
    if os.path.exists(local_parquet):
        try:
            col_count = len(pd.read_parquet(local_parquet).columns)
        except Exception as e:
            print(f"Skipping {bq_table_name}: unable to inspect local parquet {local_parquet} ({e})")
            skipped_tables.append((bq_table_name, parquet_file))
            continue
        if col_count == 0:
            print(f"Skipping {bq_table_name}: zero-column parquet {local_parquet}")
            skipped_tables.append((bq_table_name, parquet_file))
            continue

    query = f"""
    LOAD DATA OVERWRITE `{GCP_PROJECT}.{BQ_DATASET}.{bq_table_name}`
    FROM FILES (
      format = 'PARQUET',
      uris = ['{uri}']
    );
    """
    print(f"Loading {bq_table_name} from {parquet_file}...")
    job = client.query(query)
    job.result()
    loaded_tables.append((bq_table_name, parquet_file))

print(f"\nLoaded {len(loaded_tables)} parquet tables into BigQuery.")
if skipped_tables:
    print(f"Skipped {len(skipped_tables)} missing parquet tables:")
    for name, pq in skipped_tables:
        print(f"  - {name} ({pq})")
else:
    print("All expected parquet tables were present.")


## 8. Re-create the Property Graph DDL

Update the `SecGraph` Property Graph to include the new taxonomy nodes and their connective edges.

In [ ]:
# Ensure optional taxonomy tables exist before creating property graph
fallback_ddls = [
    # Taxonomy node tables
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_normalized_competitor` (id STRING, label STRING, competitor_type STRING, sector STRING, product_category STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_geographic_region` (id STRING, label STRING, description STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_market_category` (id STRING, label STRING, description STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_risk_category` (id STRING, label STRING, description STRING)",

    # Taxonomy edge tables
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_instance_of` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_subsidiary_of` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_has_risk_category` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_region` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_product_category` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_market_category` (source_node STRING, target_node STRING)",
]

for ddl in fallback_ddls:
    client.query(ddl).result()

run_bq_query("06_create_property_graph_ddl.sql")
print("\nPipeline completed successfully!")
